# **DATA STAGING**

## **1. Setup Environment**

In [1]:
import sys

print(sys.version)
print(sys.executable)

3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:12:32) [Clang 20.1.8 ]
/opt/anaconda3/envs/atoti312/bin/python


## **2. Inisialisasi Library Atoti**

In [2]:
import atoti as tt

print("ATOTI BERHASIL")

Welcome to Atoti 0.9.15!

By using this community edition, you agree with the license available at https://docs.activeviam.com/products/atoti/python-sdk/latest/eula.html.
Browse the official documentation at https://docs.activeviam.com/products/atoti/python-sdk.
Join the community at https://www.atoti.io/register.

Atoti collects telemetry data, which is used to help understand how to improve the product.
If you don't wish to send usage data, you can request a trial license at https://www.atoti.io/evaluation-license-request.

You can hide this message by setting the `ATOTI_HIDE_EULA_MESSAGE` environment variable to True.
ATOTI BERHASIL


## **3. Inspeksi Komponen Session Atoti**

In [3]:
import atoti as tt

[x for x in dir(tt) if "session" in x.lower()]

['QuerySession',
 'Session',
 'SessionConfig',
 '_session_id',
 '_session_resources',
 'query_session',
 'session',
 'session_config']

## **4. Validasi Struktur Data (Fact & Dimension)**

In [4]:
import pandas as pd

dim_date = pd.read_csv("dim_date2.csv")
dim_city = pd.read_csv("dim_city.csv")
dim_crime = pd.read_csv("dim_crime.csv")
dim_victim = pd.read_csv("dim_victim-2.csv")
dim_weapon = pd.read_csv("dim_weapon.csv")
fact_crime = pd.read_csv("fact_crime_fix3.csv")

In [5]:
print("DIM DATE :", dim_date.shape)
print("DIM CITY :", dim_city.shape)
print("DIM CRIME :", dim_crime.shape)
print("DIM VICTIM :", dim_victim.shape)
print("DIM WEAPON :", dim_weapon.shape)
print("FACT :", fact_crime.shape)

DIM DATE : (25546, 8)
DIM CITY : (29, 2)
DIM CRIME : (21, 3)
DIM VICTIM : (18, 3)
DIM WEAPON : (7, 2)
FACT : (40160, 9)


## **5. Preview Data Fact Table**

In [6]:
fact_crime.columns.tolist()

['report_number',
 'date_id',
 'city_id',
 'crime_id',
 'victim_id',
 'weapon_id',
 'police_deployed',
 'resolution_days',
 'crime_count']

In [7]:
fact_crime.head()

,report_number,date_id,city_id,crime_id,victim_id,weapon_id,police_deployed,resolution_days,crime_count
0,1,1,1,1,1,1,13,0,1
1,2,2,2,2,2,2,9,0,1
2,3,3,3,3,3,1,15,0,1
3,4,4,4,4,3,3,1,119,1
4,5,5,4,5,4,4,18,213,1


## **6. Restart Session Atoti**

In [8]:
import atoti as tt

session = tt.Session.start()

In [9]:
session.close()

## **7. Load Data into Atoti Session**

In [10]:
import atoti as tt

session = tt.Session.start()

In [11]:
date_table = session.read_pandas(
    dim_date,
    table_name="dim_date"
)

city_table = session.read_pandas(
    dim_city,
    table_name="dim_city"
)

crime_table = session.read_pandas(
    dim_crime,
    table_name="dim_crime"
)

victim_table = session.read_pandas(
    dim_victim,
    table_name="dim_victim"
)

weapon_table = session.read_pandas(
    dim_weapon,
    table_name="dim_weapon"
)

## **8. Load Fact Table**

In [12]:
fact_table = session.read_pandas(
    fact_crime,
    table_name="fact_crime"
)

In [13]:
fact_table

# session.tables

## **9. Validasi Keys Dimension Tables**

In [14]:
print(dim_date.columns.tolist())
print(dim_city.columns.tolist())
print(dim_crime.columns.tolist())
print(dim_victim.columns.tolist())
print(dim_weapon.columns.tolist())

['date_id', 'Date Reported', 'Year', 'Quarter', 'Month', 'Month Name', 'Day', 'Holiday Type']
['city_id', 'City']
['crime_id', 'Crime Description', 'Crime Domain']
['victim_id', 'Victim Gender', 'Age Group']
['weapon_id', 'Weapon Used']


## **10. Build Star Schema (Join Fact & Dimension)**

In [15]:
fact_table.join(date_table, fact_table["date_id"] == date_table["date_id"])

fact_table.join(city_table, fact_table["city_id"] == city_table["city_id"])

fact_table.join(crime_table, fact_table["crime_id"] == crime_table["crime_id"])

fact_table.join(victim_table, fact_table["victim_id"] == victim_table["victim_id"])

fact_table.join(weapon_table, fact_table["weapon_id"] == weapon_table["weapon_id"])

## **11. Create OLAP Cube**

In [16]:
cube = session.create_cube(fact_table)

cube

## **12. Inspect Cube Measures**

In [17]:
cube.measures

{'city_id.MEAN': <atoti.measure.Measure object at 0x13324c320>, 'city_id.SUM': <atoti.measure.Measure object at 0x1332b4290>, 'contributors.COUNT': <atoti.measure.Measure object at 0x1332b42f0>, 'crime_count.MEAN': <atoti.measure.Measure object at 0x1332b4aa0>, 'crime_count.SUM': <atoti.measure.Measure object at 0x1332b5280>, 'crime_id.MEAN': <atoti.measure.Measure object at 0x1332b5910>, 'crime_id.SUM': <atoti.measure.Measure object at 0x1332b59d0>, 'date_id.MEAN': <atoti.measure.Measure object at 0x1332b59a0>, 'date_id.SUM': <atoti.measure.Measure object at 0x1332b5310>, 'police_deployed.MEAN': <atoti.measure.Measure object at 0x1332b5430>, 'police_deployed.SUM': <atoti.measure.Measure object at 0x1332b5b80>, 'report_number.MEAN': <atoti.measure.Measure object at 0x1332b5c40>, 'report_number.SUM': <atoti.measure.Measure object at 0x1332b5d00>, 'resolution_days.MEAN': <atoti.measure.Measure object at 0x1332b5e20>, 'resolution_days.SUM': <atoti.measure.Measure object at 0x1332b5fa0>, 'update.TIMESTAMP': <atoti.measure.Measure object at 0x1332b6180>, 'victim_id.MEAN': <atoti.measure.Measure object at 0x1332b6270>, 'victim_id.SUM': <atoti.measure.Measure object at 0x1332b62a0>, 'weapon_id.MEAN': <atoti.measure.Measure object at 0x1332b63c0>, 'weapon_id.SUM': <atoti.measure.Measure object at 0x1332b6450>}

In [18]:
m = cube.measures

## **13. Create Custom Measures**

In [19]:
m["Total Crime"] = tt.agg.sum(fact_table["crime_count"])

m["Total Police"] = tt.agg.sum(fact_table["police_deployed"])

m["Avg Resolution Days"] = tt.agg.mean(fact_table["resolution_days"])

## **14. Inspect Measures List**

In [20]:
list(cube.measures)

['Avg Resolution Days',
 'Total Crime',
 'Total Police',
 'city_id.MEAN',
 'city_id.SUM',
 'contributors.COUNT',
 'crime_count.MEAN',
 'crime_count.SUM',
 'crime_id.MEAN',
 'crime_id.SUM',
 'date_id.MEAN',
 'date_id.SUM',
 'police_deployed.MEAN',
 'police_deployed.SUM',
 'report_number.MEAN',
 'report_number.SUM',
 'resolution_days.MEAN',
 'resolution_days.SUM',
 'update.TIMESTAMP',
 'victim_id.MEAN',
 'victim_id.SUM',
 'weapon_id.MEAN',
 'weapon_id.SUM']

## **15. Inspect Query Function**

In [22]:
import inspect

print(inspect.signature(cube.query))

(*measures: 'Measure', context: 'Context' = frozendict({}), explain: 'bool' = False, filter: 'CubeQueryFilterCondition | None' = None, include_empty_rows: 'bool' = False, include_totals: 'bool' = False, levels: 'Sequence[Level]' = (), mode: "Literal['pretty', 'raw']" = 'pretty', scenario: 'str | None' = None, **kwargs: 'Unpack[_QueryPrivateParameters]') -> 'MdxQueryResult | pd.DataFrame | object'


## **16. OLAP Query Analysis**

### **16.1 Total Crime per Year**

In [23]:
fact_year = fact_crime.merge(
    dim_date[['date_id','Year']],
    on='date_id'
)

fact_year.groupby('Year')['crime_count'].sum()

Year
2020    8747
2021    8766
2022    8759
2023    8761
2024    5127
Name: crime_count, dtype: int64

### **16.2 Top 10 City by Total Crime**

In [24]:
cube.query(
    cube.measures["Total Crime"],
    levels=[cube.levels["City"]]
).sort_values(
    by="Total Crime",
    ascending=False
).head(10)

,Total Crime
City,
Delhi,5400
Mumbai,4415
Bangalore,3588
Hyderabad,2881
Kolkata,2518
Chennai,2493
Pune,2212
Ahmedabad,1817
Jaipur,1479


### **16.3 Crime Domain vs Total Crime**

In [25]:
cube.query(
    cube.measures["Total Crime"],
    levels=[cube.levels["Crime Domain"]]
)

,Total Crime
Crime Domain,
Fire Accident,"3,825"
Other Crime,"22,948"
Traffic Fatality,"1,915"
Violent Crime,"11,472"


### **16.4 Victim Gender vs Total Crimer**

In [26]:
cube.query(
    cube.measures["Total Crime"],
    levels=[cube.levels["Victim Gender"]]
)

,Total Crime
Victim Gender,
F,"21,146"
M,"16,282"
X,"2,732"


### **16.5 Age Group vs Total Crime**

In [28]:
cube.query(
    cube.measures["Total Crime"],
    levels=[cube.levels["Age Group"]]
)

,Total Crime
Age Group,
Anak-anak,"4,369"
Dewasa,"7,770"
Dewasa Awal,"7,687"
Lansia,"9,565"
Pra Lansia,"8,116"
Remaja,"2,653"


### **16.6 Weapon Used vs Total Crime**

In [36]:
cube.query(
    cube.measures["Total Crime"],
    levels=[cube.levels["Weapon Used"]]
)

,Total Crime
Weapon Used,
Blunt Object,"5,737"
Explosives,"5,751"
Firearm,"5,643"
Knife,"5,835"
Other,"5,676"
Poison,"5,728"
Unknown,"5,790"


### **16.7 Crime Domain vs Avg Resolution Days**

In [35]:
cube.query(
    cube.measures["Avg Resolution Days"],
    levels=[cube.levels["Crime Domain"]]
)

,Avg Resolution Days
Crime Domain,
Fire Accident,23.64
Other Crime,42.13
Traffic Fatality,6.62
Violent Crime,60.82


### **16.8 City vs Avg Resolution Days**

In [34]:
fact_city = fact_crime.merge(
    dim_city,
    on="city_id"
)

fact_city.groupby(
    "City"
)["resolution_days"].mean().sort_values(
    ascending=False
)

City
Ludhiana         57.768725
Vasai            54.591160
Chennai          48.310870
Hyderabad        48.054148
Thane            47.620397
Visakhapatnam    46.788462
Lucknow          46.275412
Delhi            45.171852
Meerut           45.139241
Kolkata          45.070294
Mumbai           44.776217
Bhopal           44.063768
Srinagar         44.045822
Nashik           44.027322
Ahmedabad        42.724821
Indore           42.649499
Ghaziabad        42.430398
Rajkot           42.190625
Agra             41.602094
Patna            41.417266
Bangalore        41.347826
Jaipur           41.233266
Pune             41.172242
Surat            39.493249
Kalyan           38.194366
Faridabad        37.870056
Nagpur           37.211776
Kanpur           36.084532
Varanasi         34.397183
Name: resolution_days, dtype: float64

### **16.9 Age Categories vs Total Crime**

In [38]:
cube.query(
    cube.measures["Total Crime"],
    levels=[cube.levels["Age Group"]]
)

,Total Crime
Age Group,
Anak-anak,"4,369"
Dewasa,"7,770"
Dewasa Awal,"7,687"
Lansia,"9,565"
Pra Lansia,"8,116"
Remaja,"2,653"


### **16.10 Holiday Type vs Total Crime**

In [31]:
cube.query(
    cube.measures["Total Crime"],
    levels=[cube.levels["Holiday Type"]]
)

,Total Crime
Holiday Type,
Gandhi Jayanti,85
Hari Biasa,"39,876"
Independence Day,83
Republic Day,116


### **16.11 Holiday Type vs Crime Domain**

In [32]:
cube.query(
    cube.measures["Total Crime"],
    levels=[
        cube.levels["Holiday Type"],
        cube.levels["Crime Domain"]
    ]
)

Total Crime
Holiday Type     Crime Domain                
Gandhi Jayanti   Fire Accident             11
                 Other Crime               48
                 Traffic Fatality           2
                 Violent Crime             24
Hari Biasa       Fire Accident          3,804
                 Other Crime           22,774
                 Traffic Fatality       1,901
                 Violent Crime         11,397
Independence Day Fire Accident              5
                 Other Crime               48
                 Traffic Fatality           7
                 Violent Crime             23
Republic Day     Fire Accident              5
                 Other Crime               78
                 Traffic Fatality           5
                 Violent Crime             28

## **17. Final Data Warehouse Summary**

In [37]:
print("=== DATA WAREHOUSE BERHASIL DIBUAT ===")
print("Jumlah Record Fact :", len(fact_crime))
print("Jumlah Kota :", len(dim_city))
print("Jumlah Jenis Kejahatan :", len(dim_crime))
print("Jumlah Korban :", len(dim_victim))
print("Jumlah Senjata :", len(dim_weapon))
print("Link Dashboard :", session.link)

=== DATA WAREHOUSE BERHASIL DIBUAT ===
Jumlah Record Fact : 40160
Jumlah Kota : 29
Jumlah Jenis Kejahatan : 21
Jumlah Korban : 18
Jumlah Senjata : 7
Link Dashboard : http://localhost:49715
